In [ ]:
import os
import sys
import json
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

sys.path.append(os.path.abspath(".."))
from transformer_architecture_base import Decoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.__version__)
print(DEVICE)

In [ ]:
df = pd.read_csv("../data/scaled_test_data.csv")
df.head()

In [ ]:
with open("../data/entity_seq_pair.json", 'r') as f:
    ds = json.load(f)
ds_test = ds["test"]


In [ ]:
SEQ_LEN = 30
VAL_TIME = 12
TEST_TIME = 12
TEST_ONLY_ENITY = 45

In [ ]:
BATCH_SIZE = 384
EPOCHS = 5
LEARNING_RATE = 1e-4

HIDDEN_DIM = 128

### Data Loader

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
test_loader = DataLoader(ds_test, batch_size = BATCH_SIZE)

In [ ]:
df.columns

In [ ]:
df = df.set_index(["City_id", "Seq_id"])
df.head()

In [ ]:
col_count = len(df.columns)

model = Decoder(
    hidden_dim=HIDDEN_DIM,
    seq_len=SEQ_LEN,
    corpus_size=col_count - 4,
    n_heads=4,
    n_blocks=3,
    dropout=0.3,
    use_embedding=False,
    embedding_replacement=torch.nn.Linear(col_count, HIDDEN_DIM)
).to(DEVICE)
checkpoint = "epoch_100a"
model.load_state_dict(torch.load(f"../model/{checkpoint}.pth", map_location="cpu", weights_only=True))
model.to(DEVICE)

criterion = torch.nn.SmoothL1Loss()

print("Device:", next(model.parameters()).device)
print("Parameter Count:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Allocated Memory:", torch.cuda.memory_allocated() / 1024**3, "GB")
print("Reserved memory:", torch.cuda.memory_reserved() / 1024**3, "GB")

# Training Loop

In [ ]:
def load_tensor(entities, seq, seq_len=SEQ_LEN, df = df):
    regional_data = [] 

    for entity, sid in zip(entities, seq):
        sid = int(sid)
        entity = entity.item() if hasattr(entity, "item") else entity

        regional_data.append(torch.tensor(df.loc[entity].loc[sid:sid + seq_len].to_numpy()))


    regional_data = torch.stack(regional_data).to(DEVICE, torch.float32)
    return regional_data

In [ ]:
model.eval()
test_loss = 0
test_se = 0
test_count = 0
with torch.no_grad():
    for entities, years in tqdm(test_loader, desc=f"Test"):
        regional_data = load_tensor(entities, years)
        pred = model(regional_data[:, :-1])
        ground_truth = regional_data[:, -1, 4:]
        pred = pred[:, -1]
    

        loss = criterion(pred, ground_truth)
        test_loss += loss.item()

        test_se += ((pred - ground_truth)**2).sum().item()
        test_count += pred.shape[0]

print(f"Test Loss: {test_loss/len(test_loader)}, RMSE = {(test_se/(test_count*col_count))**0.5}")


In [ ]:
used_data = ds_test[0]

In [ ]:
pred_test = sorted(
    [x for x in ds_test if x[0] == used_data[0]],
    key=lambda x: x[1]
)
len(pred_test)

In [ ]:
result = []

with torch.no_grad():
    for entities, years in tqdm(pred_test, desc=f"Test"):
        regional_data = load_tensor([entities], [years])
        pred = model(regional_data[:, :-1])
        ground_truth = regional_data[:, -1, 4:]
        pred = pred[:, -1]
        result.append(pred)

result = torch.stack(result).squeeze(1)
result.shape

In [ ]:
from joblib import load

scaler = load("../model/scaler.pkl")

# Convert to NumPy
data = result.detach().cpu().numpy()

# Append 4 dummy columns
filler = np.zeros((data.shape[0], 4))
data = np.concatenate([filler, data], axis=1)

# Inverse transform
data = scaler.inverse_transform(data)

data = data[:, 4:]
data.shape

In [ ]:
all_seq_len = pred_test[-1][1] - pred_test[0][1] + SEQ_LEN

In [ ]:
ground_truth = load_tensor([pred_test[0][0]], [pred_test[0][1]], seq_len=all_seq_len)
ground_truth = ground_truth.squeeze(0).detach().cpu().numpy()
ground_truth = scaler.inverse_transform(ground_truth)
ground_truth = ground_truth[:, 4:]
ground_truth.shape

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
data_range = np.arange(SEQ_LEN, all_seq_len+1, 1)
gt_range = np.arange(0, all_seq_len+1, 1)

In [ ]:
colnames = df.columns[4:]

In [ ]:
fig, ax = plt.subplots(4, 5, figsize=(25, 8))

for i in range(19):
    ax[i//5, i%5].plot(data_range, data[:, i], label = "pred")
    ax[i//5, i%5].plot(gt_range, ground_truth[:, i], label = "gt")
    ax[i//5, i%5].set_title(colnames[i])
    ax[i//5, i%5].set_xlim(0, all_seq_len+11)

handles, labels = ax[0, 0].get_legend_handles_labels()

# Create a figure-level legend
fig.suptitle(f"Prediction for {pred_test[0][0]}", fontsize=20)
fig.legend(handles, labels, loc="upper center", ncol=2, fontsize=20, bbox_to_anchor=(0.5, 0.94))

plt.tight_layout(rect=[0, 0, 1, 0.88])  # Leave space for the legend
plt.show()